## 1. LOADING/TOKENIZING DATASET

In [ ]:
### LOADING DATASET
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_from_disk
from transformers import PreTrainedTokenizerFast #using transformers package for tokenization

In [14]:
def load_tokenizers(base_dir: str = "/kaggle/input/datasets/saraahmad4/dataset"):
    """
    Load the French (src) and English (tgt) BPE tokenizers
    from the provided tokenizer folders.
    """
    fr_path = f"{base_dir}/tokenizer_fr"
    en_path = f"{base_dir}/tokenizer_en"
 
    src_tokenizer = PreTrainedTokenizerFast.from_pretrained(fr_path)
    tgt_tokenizer = PreTrainedTokenizerFast.from_pretrained(en_path)
 
    # Ensure special tokens are set
    for tok in [src_tokenizer, tgt_tokenizer]:
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<pad>"})
        if tok.bos_token is None:
            tok.add_special_tokens({"bos_token": "<s>"})
        if tok.eos_token is None:
            tok.add_special_tokens({"eos_token": "</s>"})
 
    print(f"FR tokenizer vocab size : {src_tokenizer.vocab_size}")
    print(f"EN tokenizer vocab size : {tgt_tokenizer.vocab_size}")
    print(f"PAD id  — FR: {src_tokenizer.pad_token_id} | EN: {tgt_tokenizer.pad_token_id}")
    print(f"BOS id  — EN: {tgt_tokenizer.bos_token_id}")
    print(f"EOS id  — EN: {tgt_tokenizer.eos_token_id}")
 
    return src_tokenizer, tgt_tokenizer

In [15]:
def inspect_dataset(base_dir: str = "/kaggle/input/datasets/saraahmad4/dataset"):
    """
    Call this once to see what columns the Arrow dataset has.
    This tells us the exact field names for French and English.
    """
    dataset = load_from_disk(f"{base_dir}/parallel_en_fr_corpus")
    print(dataset)
    print("\nColumn names:", dataset["train"].column_names)
    print("\nFirst example:", dataset["train"][0])
    return dataset

In [16]:
class TranslationDataset(Dataset):
    """
    Wraps one split (train / validation / test) of the Arrow dataset.
 
    The Arrow dataset likely stores each example as:
        {"translation": {"en": "...", "fr": "..."}}
    OR as flat columns:
        {"en": "...", "fr": "..."}
 
    We handle both cases automatically.
    """
 
    def __init__(
        self,
        hf_split,                          # one HuggingFace dataset split
        src_tokenizer: PreTrainedTokenizerFast,
        tgt_tokenizer: PreTrainedTokenizerFast,
        src_lang: str = "text_fr",         # French = source
        tgt_lang: str = "text_en",         # English = target
        max_len: int = 32,
    ):
        self.max_len = max_len
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
 
        # ── Detect column layout ──────────────────────────────────────────
        cols = hf_split.column_names
        print(f"Dataset columns: {cols}")
 
        if "translation" in cols:
            # Nested format: {"translation": {"en": ..., "fr": ...}}
            src_sentences = [ex["translation"][src_lang] for ex in hf_split]
            tgt_sentences = [ex["translation"][tgt_lang] for ex in hf_split]
        elif src_lang in cols and tgt_lang in cols:
            # Flat format: {"en": ..., "fr": ...}
            src_sentences = list(hf_split[src_lang])
            tgt_sentences = list(hf_split[tgt_lang])
        else:
            raise ValueError(
                f"Cannot find '{src_lang}'/'{tgt_lang}' in columns: {cols}\n"
                f"Call inspect_dataset() to see the actual column names."
            )
 
        print(f"  Loaded {len(src_sentences)} sentence pairs.")
 
        # ── Tokenize everything up front ──────────────────────────────────
        self.src_ids = self._encode(src_tokenizer, src_sentences)
        self.tgt_ids = self._encode(tgt_tokenizer, tgt_sentences)
 
    def _encode(self, tokenizer, sentences):
        encoded = tokenizer(
            sentences,
            truncation=True,
            max_length=self.max_len,
            add_special_tokens=False,   # we add BOS/EOS manually below
        )
        return [torch.tensor(ids, dtype=torch.long) for ids in encoded["input_ids"]]
 
    def __len__(self):
        return len(self.src_ids)
 
    def __getitem__(self, idx):
        src = self.src_ids[idx]
 
        raw_tgt = self.tgt_ids[idx]
        bos = self.tgt_tokenizer.bos_token_id
        eos = self.tgt_tokenizer.eos_token_id
 
        # decoder_input : [BOS, t1, t2, ...]   — fed into the decoder
        # label         : [t1, t2, ..., EOS]   — what the model should predict
        decoder_input = torch.cat([torch.tensor([bos]), raw_tgt])[: self.max_len]
        label         = torch.cat([raw_tgt, torch.tensor([eos])])[: self.max_len]
 
  
        return src, decoder_input, label

In [17]:
def make_collate_fn(src_pad_id: int, tgt_pad_id: int):
    """
    Pads a batch of variable-length sequences.
 
    Returns:
        src_batch        (B, S) — padded French source
        dec_input_batch  (B, T) — padded decoder inputs  [BOS + tokens]
        label_batch      (B, T) — padded labels          [tokens + EOS]
        src_lengths      (B,)   — true source lengths (for LSTM packing)
        tgt_lengths      (B,)   — true target lengths
    """
    def collate_fn(batch):
        src_seqs, dec_inputs, labels = zip(*batch)
 
        src_lengths = torch.tensor([len(s) for s in src_seqs])
        tgt_lengths = torch.tensor([len(l) for l in labels])
 
        src_batch       = pad_sequence(src_seqs,   batch_first=True, padding_value=src_pad_id)
        dec_input_batch = pad_sequence(dec_inputs, batch_first=True, padding_value=tgt_pad_id)
        label_batch     = pad_sequence(labels,     batch_first=True, padding_value=tgt_pad_id)
 
        return src_batch, dec_input_batch, label_batch, src_lengths, tgt_lengths
 
    return collate_fn

In [18]:

def get_dataloaders(
    base_dir: str = "/kaggle/input/datasets/saraahmad4/dataset",
    batch_size: int = 32,
    max_len: int = 32,
    num_workers: int = 2,
):
    """
    Full pipeline:
        load tokenizers → load Arrow dataset → build DataLoaders
 
    Returns:
        train_loader, val_loader, test_loader,
        src_tokenizer (FR), tgt_tokenizer (EN)
    """
    src_tok, tgt_tok = load_tokenizers(base_dir)
 
    raw = load_from_disk(f"{base_dir}/parallel_en_fr_corpus")
    print(f"\nDataset splits: {list(raw.keys())}")
 
    train_ds = TranslationDataset(raw["train"],      src_tok, tgt_tok, max_len=max_len)
    val_ds   = TranslationDataset(raw["validation"], src_tok, tgt_tok, max_len=max_len)
    test_ds  = TranslationDataset(raw["test"],       src_tok, tgt_tok, max_len=max_len)
 
    collate_fn = make_collate_fn(
        src_pad_id=src_tok.pad_token_id,
        tgt_pad_id=tgt_tok.pad_token_id,
    )
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
 
    return train_loader, val_loader, test_loader, src_tok, tgt_tok

In [ ]:
# SANITY CHECK
if __name__ == "__main__":
    import sys
 
    base_dir = r"/kaggle/input/datasets/saraahmad4/dataset"
 
    print("=" * 50)
    print("Step 1: Inspecting raw dataset...")
    print("=" * 50)
    inspect_dataset(base_dir)
 
    print("\n" + "=" * 50)
    print("Step 2: Building DataLoaders...")
    print("=" * 50)
    train_loader, val_loader, test_loader, src_tok, tgt_tok = get_dataloaders(
        base_dir=base_dir, batch_size=4, max_len=32
    )
 
    print("\n" + "=" * 50)
    print("Step 3: Checking one batch...")
    print("=" * 50)
    src, dec_input, labels, src_len, tgt_len = next(iter(train_loader))
    print(f"src shape        : {src.shape}")
    print(f"dec_input shape  : {dec_input.shape}")
    print(f"labels shape     : {labels.shape}")
    print(f"src_lengths      : {src_len}")
 
    print("\nDecoding first example:")
    print(f"  FR (src)  : {src_tok.decode(src[0].tolist(), skip_special_tokens=True)}")
    print(f"  EN (label): {tgt_tok.decode(labels[0].tolist(), skip_special_tokens=True)}")
    print("\nAll good!")

Step 1: Inspecting raw dataset...
DatasetDict({
    train: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 8701
    })
    validation: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 485
    })
    test: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 486
    })
})

Column names: ['text_en', 'text_fr']

First example: {'text_en': 'i m tough .', 'text_fr': 'je suis dure .'}

Step 2: Building DataLoaders...
FR tokenizer vocab size : 3200
EN tokenizer vocab size : 3200
PAD id  — FR: 3 | EN: 3
BOS id  — EN: 1
EOS id  — EN: 2

Dataset splits: ['train', 'validation', 'test']
Dataset columns: ['text_en', 'text_fr']
  Loaded 8701 sentence pairs.
Dataset columns: ['text_en', 'text_fr']
  Loaded 485 sentence pairs.
Dataset columns: ['text_en', 'text_fr']
  Loaded 486 sentence pairs.

Step 3: Checking one batch...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


src shape        : torch.Size([4, 9])
dec_input shape  : torch.Size([4, 9])
labels shape     : torch.Size([4, 9])
src_lengths      : tensor([6, 4, 9, 7])

Decoding first example:
  FR (src)  : ▁il ▁ser a ▁bientot ▁pere ▁.
  EN (label): ▁he ▁is ▁soon ▁to ▁be ▁a ▁father ▁.

All good!


## BLEU SCORE FUNCTION + SAVING AND LOADING CHECKPOINTS

In [20]:
### BLEU SCORES FUNCTIONS
import os
import torch
from torchmetrics.text import BLEUScore

In [21]:
def save_checkpoint(
    model,
    optimizer,
    epoch: int,
    val_bleu: float,
    checkpoint_dir: str,
    model_name: str,
):
    """
    Save model + optimizer state to disk.
 
    Saves two files:
      - {model_name}_epoch{epoch}.pt  — full snapshot for resuming
      - {model_name}_best.pt          — overwritten whenever val_bleu improves
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
 
    state = {
        "epoch": epoch,
        "val_bleu": val_bleu,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
 
    epoch_path = os.path.join(checkpoint_dir, f"{model_name}_epoch{epoch}.pt")
    torch.save(state, epoch_path)
    print(f"  [Checkpoint] Saved: {epoch_path}")
 
    # Track best model separately
    best_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")
    if not os.path.exists(best_path):
        torch.save(state, best_path)
        print(f"  [Checkpoint] New best: {val_bleu:.2f} BLEU")
    else:
        prev = torch.load(best_path, map_location="cpu", weights_only=True)
        if val_bleu > prev.get("val_bleu", 0):
            torch.save(state, best_path)
            print(f"  [Checkpoint] New best: {val_bleu:.2f} BLEU "
                  f"(was {prev['val_bleu']:.2f})")

In [22]:
def load_checkpoint(model, optimizer, checkpoint_path: str, device):
    """
    Load a checkpoint. Restores model weights and optimizer state.
 
    Returns:
        start_epoch (int): epoch to resume from
        best_bleu   (float): best BLEU seen so far
    """
    if not os.path.exists(checkpoint_path):
        print(f"  [Checkpoint] No checkpoint at {checkpoint_path}, starting fresh.")
        return 0, 0.0
 
    state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state["model_state_dict"])
    optimizer.load_state_dict(state["optimizer_state_dict"])
 
    epoch    = state.get("epoch", 0)
    val_bleu = state.get("val_bleu", 0.0)
    print(f"  [Checkpoint] Loaded epoch {epoch}, val BLEU {val_bleu:.2f}")
    return epoch + 1, val_bleu

In [24]:
# One shared instance — n_gram=4 is standard BLEU-4
_bleu_metric = BLEUScore(n_gram=4)
 
def compute_bleu(
    hypotheses: list,
    references: list,
) -> float:
    """
    Compute corpus-level BLEU-4 score using torchmetrics.
 
    Args:
        hypotheses: list of predicted strings, one per sentence
                    e.g. ["the cat sat on the mat", ...]
        references: list of ground-truth strings, one per sentence
                    e.g. ["the cat is on the mat", ...]
 
    Returns:
        BLEU score as a float (0–100)
 
    Note:
        torchmetrics BLEUScore expects references as a list of lists
        (to support multiple references per sentence), so we wrap each
        reference string in a list automatically.
    """
    # torchmetrics expects: refs = [[ref1], [ref2], ...]
    wrapped_refs = [[r] for r in references]
    score = _bleu_metric(hypotheses, wrapped_refs)
    return score.item() * 100   # return as 0–100 like sacrebleu
 

In [25]:
def decode_predictions(
    token_ids_batch,       # (B, T) tensor or list of predicted token ids
    tokenizer,
    eos_token_id: int,
) -> list:
    """
    Convert a batch of token-id tensors into human-readable strings.
    Stops at EOS and strips special tokens (including the ▁ artifacts).
 
    Used by both Transformer (Member 2) and LSTM (Member 3).
    """
    sentences = []
    for ids in token_ids_batch:
        ids = ids.tolist() if hasattr(ids, "tolist") else list(ids)
        # Truncate at EOS if present
        if eos_token_id in ids:
            ids = ids[: ids.index(eos_token_id)]
        text = tokenizer.decode(ids, skip_special_tokens=True)
        sentences.append(text)
    return sentences

In [ ]:
## SANITY CHECK
if __name__ == "__main__":
    print("Testing compute_bleu...")
    hyps = ["the cat is on the mat", "hello world"]
    refs = ["the cat is on the mat", "hello world"]
    print(f"  Perfect match BLEU: {compute_bleu(hyps, refs):.2f}")   # should be 100.0
 
    hyps2 = ["the cat sat on the mat", "hi world"]
    print(f"  Imperfect BLEU:     {compute_bleu(hyps2, refs):.2f}")  # 0.00 is actually expected for such short sentences, BLEU-4 requires 4-gram matches
 
    print("utils.py OK!")

Testing compute_bleu...
  Perfect match BLEU: 100.00
  Imperfect BLEU:     0.00
utils.py OK!


## TRAINER LOOP TEMPLATE 

In [28]:
"""
### TRAINER
Shared training loop used by both the Transformer (Member 2)
and the LSTM (Member 3).
 
Each model just needs to implement a forward() that accepts:
    src, decoder_input, src_lengths → logits (B, T, vocab_size)
 
Everything else (loss, optimizer step, val loop, BLEU) lives here.
"""
import torch
import torch.nn as nn

In [29]:
class Trainer:
    """
    Generic trainer for sequence-to-sequence NMT models.
 
    Usage:
        trainer = Trainer(model, optimizer, train_loader, val_loader,
                          tgt_tokenizer, device, checkpoint_dir, model_name)
        trainer.train(num_epochs=10)
    """
 
    def __init__(
        self,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        train_loader,
        val_loader,
        tgt_tokenizer,
        device: torch.device,
        checkpoint_dir: str = "checkpoints",
        model_name: str = "model",
        pad_token_id: int = 0,
    ):
        self.model          = model.to(device)
        self.optimizer      = optimizer
        self.train_loader   = train_loader
        self.val_loader     = val_loader
        self.tgt_tokenizer  = tgt_tokenizer
        self.device         = device
        self.checkpoint_dir = checkpoint_dir
        self.model_name     = model_name
        self.pad_token_id   = pad_token_id
 
        # Cross-entropy loss; ignore PAD positions
        self.criterion = nn.CrossEntropyLoss(ignore_index=pad_token_id)
 
    # ──────────────────────────────────────────
    # Training
    # ──────────────────────────────────────────
 
    def train(self, num_epochs: int, start_epoch: int = 0):
        for epoch in range(start_epoch, num_epochs):
            train_loss = self._train_epoch(epoch)
            val_loss, val_bleu = self._val_epoch(epoch)
 
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Train loss: {train_loss:.4f} | "
                f"Val loss: {val_loss:.4f} | "
                f"Val BLEU: {val_bleu:.2f}"
            )
 
            save_checkpoint(
                self.model, self.optimizer,
                epoch, val_bleu,
                self.checkpoint_dir, self.model_name,
            )
 
    def _train_epoch(self, epoch: int) -> float:
        self.model.train()
        total_loss = 0.0
 
        for batch_idx, (src, dec_input, labels, src_lengths, _) in enumerate(self.train_loader):
            src        = src.to(self.device)
            dec_input  = dec_input.to(self.device)
            labels     = labels.to(self.device)
            src_lengths = src_lengths.to(self.device)
 
            self.optimizer.zero_grad()
 
            # --- Forward pass ---
            # model.forward() must return logits of shape (B, T, vocab_size)
            logits = self.model(src, dec_input, src_lengths)
 
            # Flatten for cross-entropy: (B*T, vocab) vs (B*T,)
            loss = self.criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
            )
 
            loss.backward()
 
            # Gradient clipping prevents exploding gradients (common in RNNs)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
 
            self.optimizer.step()
            total_loss += loss.item()
 
            if (batch_idx + 1) % 100 == 0:
                print(f"  Step {batch_idx+1} | Loss: {loss.item():.4f}")
 
        return total_loss / len(self.train_loader)
 
    # ──────────────────────────────────────────
    # Validation
    # ──────────────────────────────────────────
 
    def _val_epoch(self, epoch: int):
        self.model.eval()
        total_loss  = 0.0
        hypotheses  = []
        references  = []
 
        eos_id = self.tgt_tokenizer.eos_token_id
        bos_id = self.tgt_tokenizer.bos_token_id
 
        with torch.no_grad():
            for src, dec_input, labels, src_lengths, _ in self.val_loader:
                src        = src.to(self.device)
                dec_input  = dec_input.to(self.device)
                labels     = labels.to(self.device)
                src_lengths = src_lengths.to(self.device)
 
                logits = self.model(src, dec_input, src_lengths)
 
                loss = self.criterion(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1),
                )
                total_loss += loss.item()
 
                # Greedy decoding for BLEU (beam search added by Member 4)
                pred_ids = logits.argmax(dim=-1)   # (B, T)
                hyps = decode_predictions(pred_ids, self.tgt_tokenizer, eos_id)
                refs = decode_predictions(labels,   self.tgt_tokenizer, eos_id)
 
                hypotheses.extend(hyps)
                references.extend(refs)
 
        avg_loss = total_loss / len(self.val_loader)
        bleu     = compute_bleu(hypotheses, references)
        return avg_loss, bleu